In [ ]:
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display
from scipy import sparse
from scipy.sparse.linalg import factorized
from statsmodels.stats.multitest import multipletests

from analysis.src.surrogates import (
    calibrate_rho, conditioned_residuals, moran_i, repaired_queen_weights, residualise
)
from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import ADJUSTMENT_SETS, OUTCOMES, RANDOM_STATE
from paths import paths

In [ ]:
OUT = experiment_paths('spatial/conditional_null')
DISEASES = OUTCOMES
N_SURROGATES = 999

data = load_data()
walkability_sd = data.walkability_score.std(ddof = 1)

In [ ]:
# Load geographic inputs
data = gpd.read_parquet(paths.analysis.experiments['geography/gradients_and_shortfalls'].spatial_parquet)

gradient_summary = json.loads(
    (paths.analysis.experiments['geography/gradients_and_shortfalls'].summary_json).read_text()
)
gradient_formula = gradient_summary['gradient_formula']

In [ ]:
def _conditional_coefficients(null_data, outcome, covariates, surrogates):
    observed_walkability = null_data['walkability_score'].to_numpy()
    nuisance = np.column_stack([np.ones(len(null_data)), null_data[covariates].to_numpy()])
    nuisance_q, _ = np.linalg.qr(nuisance, mode = 'reduced')

    y_residual = residualise(null_data[outcome].to_numpy(), nuisance_q)
    observed_x_residual = residualise(observed_walkability, nuisance_q)
    observed_beta_raw = (observed_x_residual @ y_residual) / (observed_x_residual @ observed_x_residual)

    surrogate_x_residuals = residualise(surrogates, nuisance_q)
    beta_raw = (surrogate_x_residuals.T @ y_residual) / np.sum(surrogate_x_residuals**2, axis = 0)

    return observed_beta_raw, beta_raw

In [ ]:
def _coefficient_summary(observed_beta, betas):
    n_surrogates = len(betas)
    null_median = float(np.median(betas))
    signed_displacement = observed_beta - null_median
    empirical_p = (1 + (np.abs(betas - null_median) >= abs(signed_displacement)).sum()) / (1 + n_surrogates)

    return {
        'observed_coefficient_per_sd': observed_beta,
        'null_coefficient_mean': float(betas.mean()),
        'null_coefficient_median': null_median,
        'null_coefficient_sd': float(betas.std(ddof = 1)),
        'null_coefficient_ci_lower': float(np.quantile(betas, 0.025)),
        'null_coefficient_ci_upper': float(np.quantile(betas, 0.975)),
        'observed_signed_displacement_from_null_median': signed_displacement,
        'observed_absolute_displacement_from_null_median': abs(signed_displacement),
        'observed_null_percentile': float((betas <= observed_beta).mean()),
        'empirical_two_sided_p': empirical_p
    }

In [ ]:
def run_conditional_null_case(
    null_data, outcome, covariates, gradient_formula, walkability_sd, adjustment,
    n_surrogates, rng, weights, rho = None, calibration_expected_i = None,
):
    weight_matrix = weights.sparse.tocsc()
    identity = sparse.identity(len(null_data), format = 'csc')

    gradient_fit = smf.ols(gradient_formula, data = null_data).fit()
    gradient_design = np.asarray(gradient_fit.model.exog)
    gradient_q, _ = np.linalg.qr(gradient_design, mode = 'reduced')
    fixed_gradient = gradient_fit.fittedvalues.to_numpy()

    observed_residual = gradient_fit.resid.to_numpy()

    if rho is None:
        rho, calibration_expected_i, target_residual_i, calibration = calibrate_rho(
            identity, weight_matrix, observed_residual, gradient_q, weights, rng
        )
    else:
        target_residual_i = moran_i(observed_residual, weights)
        calibration = pd.DataFrame()

    solver = factorized(identity - rho * weight_matrix)
    fields = solver(rng.standard_normal((len(null_data), n_surrogates)))
    residual_surrogates = conditioned_residuals(fields, observed_residual, gradient_q)
    surrogates = fixed_gradient[:, None] + residual_surrogates

    observed_beta_raw, beta_raw = _conditional_coefficients(null_data, outcome, covariates, surrogates)
    betas = beta_raw * walkability_sd

    simulation_rows = pd.DataFrame(
        {
            'adjustment': adjustment,
            'outcome': outcome,
            'simulation': np.arange(1, n_surrogates + 1),
            'coefficient_per_sd': betas
        }
    )
    
    summary = {
        'adjustment': adjustment,
        'outcome': outcome,
        'n_lsoas': len(null_data),
        **_coefficient_summary(float(observed_beta_raw * walkability_sd), betas),
        'gradient_r_squared': gradient_fit.rsquared,
        'car_rho': rho,
        'target_residual_moran_i': target_residual_i,
        'calibrated_expected_residual_moran_i': calibration_expected_i,
        'n_surrogates': n_surrogates
    }

    return simulation_rows, summary, calibration

In [ ]:
case_number = 0
graph_cache, calibration_cache = {}, {}
null_rows, null_summary_rows, calibration_rows, island_link_rows = [], [], [], []
for adjustment, covariates in ADJUSTMENT_SETS.items():
    for outcome in DISEASES:
        case_number += 1

        null_data = data.loc[complete_case_mask(data, outcome, covariates)].copy().reset_index(drop = True)
        sample_key = tuple(null_data['LSOA21CD'])

        if sample_key not in graph_cache:
            graph_cache[sample_key] = repaired_queen_weights(null_data)

        weights, island_links = graph_cache[sample_key]
        island_link_rows.extend(island_links.assign(adjustment = adjustment, outcome = outcome).to_dict('records'))

        rng = np.random.default_rng(np.random.SeedSequence([RANDOM_STATE, case_number]))
        cached = calibration_cache.get(sample_key)

        simulation_rows, summary, calibration = run_conditional_null_case(
            null_data = null_data,
            outcome = outcome,
            covariates = covariates,
            gradient_formula = gradient_formula,
            walkability_sd = walkability_sd,
            adjustment = adjustment,
            n_surrogates = N_SURROGATES,
            rng = rng,
            weights = weights,
            rho = None if cached is None else cached['rho'],
            calibration_expected_i = None if cached is None else cached['expected_i'],
        )

        if cached is None:
            calibration_cache[sample_key] = {
                'rho': summary['car_rho'],
                'expected_i': summary['calibrated_expected_residual_moran_i']
            }

        if not calibration.empty:
            calibration_rows.extend(calibration.assign(adjustment = adjustment, outcome = outcome).to_dict('records'))

        null_rows.extend(simulation_rows.to_dict('records'))
        null_summary_rows.append(summary)

        print(
            f'{case_number:02d}/{len(ADJUSTMENT_SETS) * len(DISEASES)} {adjustment} | {outcome}: '
            f'p = {summary["empirical_two_sided_p"]:.4f}, rho = {summary["car_rho"]:.4f}'
        )

In [ ]:
conditional_null_results = pd.DataFrame(null_rows)
conditional_null_summary = pd.DataFrame(null_summary_rows)

for adjustment, index in conditional_null_summary.groupby('adjustment').groups.items():
    rejected, q_values, _, _ = multipletests(
        conditional_null_summary.loc[index, 'empirical_two_sided_p'], method = 'fdr_bh'
    )
    conditional_null_summary.loc[index, 'empirical_q_value'] = q_values
    conditional_null_summary.loc[index, 'significant_fdr_05'] = rejected

calibration_results = pd.DataFrame(calibration_rows)
island_link_results = pd.DataFrame(island_link_rows).drop_duplicates()

conditional_null_results.to_csv(OUT.conditional_null_draws_csv, index = False)
conditional_null_summary.to_csv(OUT.conditional_null_summary_csv, index = False)
calibration_results.to_csv(OUT.null_calibration_csv, index = False)
island_link_results.to_csv(OUT.island_links_csv, index = False)

display(island_link_results)
display(conditional_null_summary.round(4))